In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [3]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

12.43

In [ ]:
def calculate_heikin_ashi(df):
    ha_close = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    ha_open = (ha_close.shift(1) + ha_close.shift(1)) / 2
    ha_high = df[['high', 'open', 'close']].max(axis=1)
    ha_low = df[['low', 'open', 'close']].min(axis=1)

    return pd.DataFrame({'ha_open': ha_open, 'ha_high': ha_high, 'ha_low': ha_low, 'ha_close': ha_close, 'time':df.time})

In [ ]:
import numpy as np
import pandas as pd
import pandas_ta as pdt

def supertrend(factor, atr_length, high, low, close):
    atr = pdt.atr(high, low, close, atr_length)
    basic_upper_band = (high + low) / 2 + factor * atr
    basic_lower_band = (high + low) / 2 - factor * atr
    bullish_signal = close > basic_upper_band
    bearish_signal = close < basic_lower_band
    bullish_supertrend = np.full_like(close, np.nan)
    bearish_supertrend = np.full_like(close, np.nan)

    for i in range(1, len(close)):
        if bullish_signal[i] or (bullish_supertrend[i-1] and close[i-1] > basic_upper_band[i-1]):
            bullish_supertrend[i] = max(basic_upper_band[i], bullish_supertrend[i-1])
        else:
            bullish_supertrend[i] = basic_upper_band[i]

        if bearish_signal[i] or (bearish_supertrend[i-1] and close[i-1] < basic_lower_band[i-1]):
            bearish_supertrend[i] = min(basic_lower_band[i], bearish_supertrend[i-1])
        else:
            bearish_supertrend[i] = basic_lower_band[i]

    direction = np.where(close > bullish_supertrend, 1, np.where(close < bearish_supertrend, -1, 0))
    supertrend = np.where(direction == 1, bullish_supertrend, bearish_supertrend)
    
    return supertrend, direction

# Example usage:
# Assuming df is your DataFrame containing OHLC data
# Replace this with your actual DataFrame
# Example:
# df = pd.DataFrame({'open': [...], 'high': [...], 'low': [...], 'close': [...]})

# Convert input parameters from Pine Script to Python
# factor = 3.0
# atr_length = 10




In [ ]:
pdt.atr?

In [4]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [5]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 12)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertren
    return rates_frame

In [ ]:
a = get_values('GBPUSD', 1000, 50, 'D1')

In [ ]:
a

In [ ]:
type(direction)

In [4]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [5]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [ ]:
def calculate_rsi(data, window):
    # Calculate the differences in the data
    delta = data.diff()

    # Make the positive gains (up) and negative gains (down) Series
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)

    # Calculate the average gain and average loss
    avg_gain = gain.rolling(window=window, min_periods=1).mean()
    avg_loss = loss.rolling(window=window, min_periods=1).mean()

    # Calculate the RS (Relative Strength)
    rs = avg_gain / avg_loss

    # Calculate the RSI
    rsi = 100 - (100 / (1 + rs))

    return rsi


In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
# a = get_values('USDJPY', 10000, 100,12, 'H1')
# a = get_values(symbol, 10000, 100,21, 'H4')
# a = get_values(symbol, 10000, 50,21, 'D1')
# a = get_values(symbol, 10000, 50, 12, 'H1')
a = get_values(symbol, 10000, 50,9, 'M30')

lot = 0.1
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

# 
print("here")
print(lot)
for i in range(1, len(a)):
    if check==0:
#         if a.iloc[i-1].close <= a.iloc[i-1].sma and  a.iloc[i-1].open >= a.iloc[i-1].sma:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=1
            
        if a.iloc[i].rsi < 25 and  a.iloc[i-1].rsi >= 50 and a.iloc[i]:
            print(f"{a.iloc[i].name}--- {a.iloc[i].open} --{a.iloc[i].rsi}--{a.iloc[i].close}")
            buy_price = a.iloc[i].open
            diff = abs(a.iloc[i].close - a.iloc[i].open)/2
            check=2
#             continue
#     if check==1:
#         sell_price = a.iloc[i].close
#         pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
# #         print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#         if a.iloc[i].high >= a.iloc[i].sma and  (a.iloc[i].high - a.iloc[i].sma) >= 0.00300:
#             pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma+0.00300, mt5.ORDER_TYPE_SELL)
#             print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
#             if pp1<=-10:
#                 profit.append(-10)
#             else:
#                 profit.append(pp1)
#             check=0
#         elif pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
#             check = 0
            
    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#         profit.append(pp)
#         check=0
#         if a.iloc[i].low <= a.iloc[i].sma and  (a.iloc[i].sma - a.iloc[i].low) >= 0.00300:
#             pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma-0.00300, mt5.ORDER_TYPE_BUY)
#             print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
#             if pp1<=-20:
#                 profit.append(-20)
#             else:
#                 profit.append(pp1)
#             check=0
#         elif pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if a.iloc[i].rsi < a.iloc[i-1].rsi or pp < 0.0:
            print(f"pp- {pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#             print(f"Diff--> {diff}")
#             if pp <0.0 and sell_price < (a.iloc[i].open-diff):
#                 pp1 = price_action(symbol, lot, buy_price, a.iloc[i].close-diff, mt5.ORDER_TYPE_BUY)
#                 print(f"pp1- {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 profit.append(pp1)
#                 check = 0
#             elif sell_price < (a.iloc[i].open-diff):
# #             else:
#                 print(f"pp- {pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 profit.append(pp)
#                 check = 0

In [9]:
def get_values(symbol, size, smaa=50,r=14, t='M30'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H6':mt5.TIMEFRAME_H6,'H8':mt5.TIMEFRAME_H8, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['rsi'] = get_rsi(rates_frame['close'], r)
#     rates_frame['rsi'] = get_rsi(rates_frame['close'], r)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()

    rates_frame['sma1'] = rates_frame['close'].rolling(window=9).mean()
    rates_frame['sma2'] = rates_frame['close'].rolling(window=21).mean()
    rates_frame = rates_frame[rates_frame['sma1'].notna()]
    rates_frame = rates_frame[rates_frame['sma2'].notna()]

    # Calculate Supertren
    return rates_frame

In [91]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "GBPJPY"
a = 0
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
# a = get_values('USDJPY', 10000, 100,12, 'H1')
# a = get_values(symbol, 10000, 100,21, 'H4')
# a = get_values(symbol, 10000, 50,21, 'D1')
# a = get_values(symbol, 10000, 50, 12, 'H1')

#BTCUSD
# a = get_values(symbol, 10000, 50,21, 'M30')
#GBPJPY
a = get_values(symbol, 10000, 100,21, 'H1')
# a = get_values('GBPJPY', 10000, 50,21, 'H2')
# a = get_values('GBPJPY', 10000, 50,21, 'H4')
# a = get_values('GBPJPY', 10000, 50,21, 'H6')


lot = 0.1
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

# 
print("here")
print(lot)
for i in range(1, len(a)-1):
    if check==0:
        if a.iloc[i].sma1 < a.iloc[i].sma2 and a.iloc[i].close < a.iloc[i].open and a.iloc[i].close < a.iloc[i].sma1 and a.iloc[i].open > a.iloc[i].sma1 and a.iloc[i].close < a.iloc[i].sma:
            print(f"{a.iloc[i].name}--- {a.iloc[i].open} --{a.iloc[i].rsi}--{a.iloc[i].close}")
            print('='*20)
            buy_price = a.iloc[i].close
            p = []
            check=1
            continue
            
#         if a.iloc[i].sma1 > a.iloc[i].sma2 and a.iloc[i].close > a.iloc[i].open and a.iloc[i].close > a.iloc[i].sma1 and a.iloc[i].open < a.iloc[i].sma1 and a.iloc[i].close > a.iloc[i].sma:
#             print(f"{a.iloc[i].name}--- {a.iloc[i].open} --{a.iloc[i].rsi}--{a.iloc[i].close}")
#             print('='*20)
#             buy_price = a.iloc[i].close
#             diff = abs(a.iloc[i].close - a.iloc[i].open)/2
#             check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 1.40
        p.append(a.iloc[i-1].rsi)
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}-- RSI {a.iloc[i].rsi} --{a.iloc[i].name}")
        if a.iloc[i].close > a.iloc[i].sma1 or a.iloc[i].rsi<=25.5:
            try:
                conti.append([pp, p[1], p[0], a.iloc[i].name])
            except:
                pass
            profit.append(pp)
#             print(f"RSI --- {abs(p[0])}")
            check=0
            print('-'*20)
#         if a.iloc[i-1].close>a.iloc[i-1].sma1 and a.iloc[i].close<a.iloc[i].sma1:
#             if a.iloc[i+1].close >= a.iloc[i+1].open:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_SELL) - 2.50
# #             elif a.iloc[i+1].close < a.iloc[i].sma2:
# #                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 2.50
#             else:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].high, mt5.ORDER_TYPE_SELL) - 2.50
#             print(f"TEST--  {pp1}  --")
# #             profit.append(pp1)
#             if pp1<-8:
#                 profit.append(-8)
#             else:
#                 profit.append(pp1)
            
    elif check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) -1.4
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}-- RSI {a.iloc[i].rsi} --{a.iloc[i].name}")
#         profit.append(pp)
#         check=0
        if a.iloc[i].close < a.iloc[i].sma2 or a.iloc[i].rsi>=78.5:
            profit.append(pp)
            check=0
            print('-'*20)
#         if a.iloc[i-1].close<a.iloc[i-1].sma1 and a.iloc[i].close>a.iloc[i].sma1:
#             if a.iloc[i+1].close >= a.iloc[i+1].open:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - 1.4
# #             elif a.iloc[i+1].close < a.iloc[i].sma2:
# #                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 2.50
#             else:
#                 pp1 = price_action(symbol, lot, a.iloc[i].close, a.iloc[i+1].low, mt5.ORDER_TYPE_BUY) - 1.4
#             print(f"TEST--  {pp1}  --")
# #             profit.append(pp1)
#             if pp1<-8:
#                 profit.append(-8)
#             else:
#                 profit.append(pp1)

here
0.1
2022-12-21 16:00:00--- 160.614 --39.841144160724504--159.932
6.699999999999999--164.44653---159.806-- RSI 39.168510936921315 --2022-12-21 17:00:00
8.69--164.36379---159.775-- RSI 38.99842280782745 --2022-12-21 18:00:00
13.06--164.28078000000002---159.707-- RSI 38.61223513852843 --2022-12-21 19:00:00
12.54--164.19539---159.715-- RSI 38.687237099732656 --2022-12-21 20:00:00
15.179999999999998--164.1111---159.674-- RSI 38.434542660301815 --2022-12-21 21:00:00
-3.84--164.02760999999998---159.97-- RSI 41.33906107098579 --2022-12-21 22:00:00
--------------------
2022-12-22 02:00:00--- 159.854 --40.20744902300735--159.806
4.449999999999999--163.62814---159.715-- RSI 39.50098640581742 --2022-12-22 03:00:00
7.4--163.54725---159.669-- RSI 39.13600986628901 --2022-12-22 04:00:00
-3.46--163.46643---159.838-- RSI 41.23072752936208 --2022-12-22 05:00:00
--------------------
2022-12-22 06:00:00--- 159.839 --39.999097268851216--159.694
-11.68--163.30879---159.854-- RSI 42.019725468316146 --20

2023-03-24 02:00:00--- 160.708 --37.61479888884294--160.306
8.37--161.35325---160.154-- RSI 36.11646969515375 --2023-03-24 03:00:00
11.84--161.34417---160.1-- RSI 35.587674406897094 --2023-03-24 04:00:00
12.86--161.33294---160.084-- RSI 35.42630408977473 --2023-03-24 05:00:00
29.310000000000002--161.31436---159.828-- RSI 32.91860861593648 --2023-03-24 06:00:00
27.19--161.30139---159.861-- RSI 33.55521809915406 --2023-03-24 07:00:00
17.94--161.29083---160.005-- RSI 36.32397507082724 --2023-03-24 08:00:00
28.93--161.28194---159.834-- RSI 34.529893782520816 --2023-03-24 09:00:00
69.35--161.26833---159.205-- RSI 28.99814125567532 --2023-03-24 10:00:00
116.13--161.24611---158.477-- RSI 24.27257929463532 --2023-03-24 11:00:00
--------------------
2023-03-28 12:00:00--- 161.131 --48.263736612382786--160.811
-28.13--160.88713---161.227-- RSI 54.035701010579395 --2023-03-28 13:00:00
--------------------
2023-04-05 02:00:00--- 164.522 --42.772099466902496--164.181
0.08000000000000007--164.361620

2023-07-11 15:00:00--- 181.308 --39.976051479132295--181.167
-36.68--182.64858---181.716-- RSI 48.98466680946389 --2023-07-11 16:00:00
--------------------
2023-07-11 18:00:00--- 181.327 --43.255209280967414--181.258
-6.67--182.58534---181.34-- RSI 44.526390796160555 --2023-07-11 19:00:00
--------------------
2023-07-11 20:00:00--- 181.34 --44.25977522142944--181.319
-18.24--182.54241000000002---181.581-- RSI 48.31403823764346 --2023-07-11 21:00:00
--------------------
2023-07-18 15:00:00--- 181.224 --43.233481705398496--180.762
-37.19--181.02037---181.319-- RSI 50.498577809743644 --2023-07-18 16:00:00
--------------------
2023-07-18 23:00:00--- 181.174 --45.770705880485--180.961
-1.0799999999999998--181.11466000000001---180.956-- RSI 45.70381751531238 --2023-07-19 00:00:00
-12.77--181.12883000000002---181.138-- RSI 48.576046849533824 --2023-07-19 01:00:00
-16.24--181.14189---181.192-- RSI 49.40978186300479 --2023-07-19 02:00:00
--------------------
2023-07-19 03:00:00--- 181.192 --47.

-11.620000000000001--182.30427---181.696-- RSI 48.837239375961104 --2023-09-25 12:00:00
--------------------
2023-09-26 05:00:00--- 181.746 --45.544614131782026--181.59
-0.6299999999999999--182.05155---181.578-- RSI 45.191740360569206 --2023-09-26 06:00:00
-9.11--182.03824---181.71-- RSI 49.69355837930771 --2023-09-26 07:00:00
--------------------
2023-09-26 08:00:00--- 181.71 --48.45876514525072--181.671
7.15--182.00838---181.538-- RSI 44.49940772162996 --2023-09-26 09:00:00
27.64--181.98979---181.219-- RSI 36.90540595712907 --2023-09-26 10:00:00
13.83--181.97280999999998---181.434-- RSI 43.704167160219704 --2023-09-26 11:00:00
16.080000000000002--181.955---181.399-- RSI 42.913756027760016 --2023-09-26 12:00:00
15.37--181.94145---181.41-- RSI 43.25243743220367 --2023-09-26 13:00:00
22.57--181.92341999999996---181.298-- RSI 40.67268873126067 --2023-09-26 14:00:00
16.59--181.90575---181.391-- RSI 43.60533101890086 --2023-09-26 15:00:00
14.469999999999999--181.88913---181.424-- RSI 44.62

2023-11-28 14:00:00--- 187.507 --45.30612793147463--187.366
-13.93--187.46648000000002---187.561-- RSI 49.12270494447087 --2023-11-28 15:00:00
--------------------
2023-11-29 06:00:00--- 187.165 --42.936401103024--186.999
-5.32--187.54853000000003---187.06-- RSI 44.28026724497176 --2023-11-29 07:00:00
--------------------
2023-11-29 10:00:00--- 187.09 --43.084167021845005--186.991
-9.5--187.57327999999998---187.117-- RSI 46.09193402271024 --2023-11-29 11:00:00
--------------------
2023-11-30 07:00:00--- 186.752 --41.20517827330769--186.704
-2.4299999999999997--187.50119999999998---186.72-- RSI 41.67339629463876 --2023-11-30 08:00:00
--------------------
2023-11-30 09:00:00--- 186.722 --40.57085252056234--186.668
17.17--187.48512---186.379-- RSI 35.144823524066766 --2023-11-30 10:00:00
11.84--187.47735999999998---186.462-- RSI 37.65908642739632 --2023-11-30 11:00:00
4.25--187.47025000000002---186.58-- RSI 41.06944309508948 --2023-11-30 12:00:00
-2.88--187.46455999999998---186.691-- RSI 

2024-03-06 23:00:00--- 190.246 --43.04977714932455--190.151
4.380000000000001--190.36692---190.061-- RSI 41.08685172700444 --2024-03-07 00:00:00
2.2--190.37636000000003---190.095-- RSI 42.133464617943126 --2024-03-07 01:00:00
29.96--190.38058---189.663-- RSI 34.060711178004325 --2024-03-07 02:00:00
60.480000000000004--190.37956999999997---189.188-- RSI 27.8910822502353 --2024-03-07 03:00:00
49.43--190.37981---189.36-- RSI 32.53723622173118 --2024-03-07 04:00:00
53.15--190.37803---189.302-- RSI 31.811501936621525 --2024-03-07 05:00:00
56.95--190.37385999999998---189.243-- RSI 31.071267529801602 --2024-03-07 06:00:00
55.72--190.36690000000002---189.262-- RSI 31.609379343784042 --2024-03-07 07:00:00
85.72999999999999--190.35559999999998---188.795-- RSI 26.308774473236213 --2024-03-07 08:00:00
110.66--190.34080999999998---188.407-- RSI 22.95124485016204 --2024-03-07 09:00:00
--------------------
2024-03-08 11:00:00--- 189.494 --38.26900218667881--188.731
-7.119999999999999--190.10238---188

2024-07-11 15:00:00--- 208.004 --24.298404312407797--205.153
6.76--206.51832000000002---205.026-- RSI 23.56569241380285 --2024-07-11 16:00:00
--------------------
2024-07-12 02:00:00--- 205.654 --30.44488210714944--204.565
-52.03--206.41739---205.353-- RSI 40.83946566039515 --2024-07-12 03:00:00
--------------------
2024-07-12 08:00:00--- 205.426 --41.49553813134169--205.356
-3.46--206.40609---205.388-- RSI 41.91747945229116 --2024-07-12 09:00:00
--------------------
2024-07-15 03:00:00--- 205.249 --41.80331669662784--205.006
3.7399999999999998--206.27277999999998---204.926-- RSI 40.96137297539202 --2024-07-15 04:00:00
0.98--206.26301---204.969-- RSI 41.624913764709625 --2024-07-15 05:00:00
2.39--206.2511---204.947-- RSI 41.37510137660122 --2024-07-15 06:00:00
-0.30999999999999983--206.24027999999998---204.989-- RSI 42.07199280267401 --2024-07-15 07:00:00
2.7100000000000004--206.22870999999998---204.942-- RSI 41.49244414890302 --2024-07-15 08:00:00
-4.68--206.21787---205.057-- RSI 43.4

In [92]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

-1034.0300000000002
Total negative sm -->-2550.6500000000005
Total negative -->180
Total positive sm -->1516.6200000000003
Total positive -->49
Length 229


In [93]:
profit.sort()

In [94]:
profit

[-52.03,
 -50.559999999999995,
 -44.58,
 -44.26,
 -41.879999999999995,
 -41.82,
 -38.089999999999996,
 -37.19,
 -36.68,
 -36.61,
 -36.16,
 -35.71,
 -34.809999999999995,
 -32.76,
 -31.99,
 -31.209999999999997,
 -30.7,
 -29.93,
 -29.479999999999997,
 -28.77,
 -28.13,
 -28.13,
 -27.939999999999998,
 -27.75,
 -26.59,
 -24.79,
 -24.279999999999998,
 -23.95,
 -22.99,
 -22.599999999999998,
 -22.029999999999998,
 -21.639999999999997,
 -20.869999999999997,
 -20.549999999999997,
 -20.29,
 -20.229999999999997,
 -19.97,
 -19.2,
 -18.939999999999998,
 -18.88,
 -18.689999999999998,
 -18.56,
 -18.36,
 -18.24,
 -17.91,
 -17.849999999999998,
 -17.72,
 -17.59,
 -17.59,
 -17.459999999999997,
 -17.21,
 -16.95,
 -16.95,
 -16.759999999999998,
 -16.56,
 -16.31,
 -16.24,
 -16.18,
 -16.11,
 -15.540000000000001,
 -15.22,
 -15.15,
 -15.15,
 -15.09,
 -14.96,
 -14.89,
 -14.89,
 -14.89,
 -14.77,
 -14.77,
 -14.57,
 -14.120000000000001,
 -13.99,
 -13.93,
 -13.870000000000001,
 -13.870000000000001,
 -13.87000000000000

In [95]:
conti.sort()
conti

[[-50.559999999999995,
  37.65947659177231,
  32.036022682514755,
  Timestamp('2023-07-28 09:00:00')],
 [-44.26,
  26.959559874486885,
  24.850778232974505,
  Timestamp('2023-12-08 05:00:00')],
 [-41.82,
  36.80438239891038,
  36.59501029922592,
  Timestamp('2023-10-04 11:00:00')],
 [-36.61,
  32.79883476416225,
  30.971304249072105,
  Timestamp('2023-02-03 11:00:00')],
 [-36.16,
  33.74571245658207,
  33.75824961297691,
  Timestamp('2023-02-10 16:00:00')],
 [-34.809999999999995,
  45.482715251992865,
  43.197430600952075,
  Timestamp('2023-10-17 17:00:00')],
 [-32.76,
  34.97808076958617,
  35.71072479721644,
  Timestamp('2023-05-11 11:00:00')],
 [-30.7,
  41.11107033372592,
  42.28698752628804,
  Timestamp('2023-03-16 09:00:00')],
 [-29.93,
  42.25061477251085,
  41.4757336143426,
  Timestamp('2023-04-14 15:00:00')],
 [-29.479999999999997,
  39.976609762069465,
  38.26900218667881,
  Timestamp('2024-03-08 16:00:00')],
 [-28.77,
  42.07527288480749,
  44.11370427207357,
  Timestamp('2

In [ ]:
a.iloc[-2].name == a.iloc[-2].name

In [ ]:
symbol = "BTCUSD"
a = get_values(symbol, 20000, 150, 'M15')

,open,high,low,close,rsi,sma,sma1,sma2
time,,,,,,,,
2022-12-08 22:00:00,167.319,167.336,167.208,167.281,65.971484,NaN,166.996333,166.870667
2022-12-08 23:00:00,167.281,167.298,167.181,167.222,62.989874,NaN,167.074222,166.895533
2022-12-09 00:00:00,167.188,167.188,166.990,167.110,57.784399,NaN,167.103889,166.915933
2022-12-09 01:00:00,167.110,167.184,167.009,167.157,59.267596,NaN,167.124556,166.951867
2022-12-09 02:00:00,167.157,167.399,167.097,167.229,61.446384,NaN,167.174667,166.985267


In [ ]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M10':mt5.TIMEFRAME_M10, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)

    rates_frame = calculate_heikin_ashi(rates_frame)
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['sma'] = rates_frame['close'].rolling(window=50).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]

    return rates_frame

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 150, 'M10')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].ha_open < a.iloc[j].ha_close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].ha_close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].ha_open
            check=1
            
        if a.iloc[i-1].ha_close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].ha_open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].ha_close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].ha_close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].ha_close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].ha_close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "GBPUSD"
# GBPUSD = 0.00100
# EURUSD = 0.00050
# USDJPY = 0.491
a = get_values(symbol, 10000, 50, 'D1')
lot = 0.1
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

# 
print("here")
print(lot)
for i in range(1, len(a)):
    if check==0:
#         if a.iloc[i-1].close <= a.iloc[i-1].sma and  a.iloc[i-1].open >= a.iloc[i-1].sma:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=1
            
        if a.iloc[i-1].close >= a.iloc[i-1].sma and a.iloc[i-1].open <= a.iloc[i-1].sma:
            print(f"{a.iloc[i].name}--- {a.iloc[i-1].open} --{a.iloc[i-1].sma}--{a.iloc[i-1].close}")
            buy_price = a.iloc[i].open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
#         print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if a.iloc[i].high >= a.iloc[i].sma and  (a.iloc[i].high - a.iloc[i].sma) >= 0.00300:
            pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma+0.00300, mt5.ORDER_TYPE_SELL)
            print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
            if pp1<=-10:
                profit.append(-10)
            else:
                profit.append(pp1)
            check=0
        elif pp >= 0.0:
            print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
            
    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
#         print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if a.iloc[i].low <= a.iloc[i].sma and  (a.iloc[i].sma - a.iloc[i].low) >= 0.00300:
            pp1 = price_action(symbol, lot, buy_price, a.iloc[i].sma-0.00300, mt5.ORDER_TYPE_BUY)
            print(f"PP1 ---> {pp1}--{ a.iloc[i].sma}---{a.iloc[i].close}-{a.iloc[i].name}")
            if pp1<=-20:
                profit.append(-20)
            else:
                profit.append(pp1)
            check=0
        elif pp >= 0.0:
            print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0